# FAIL_TO_PASS / PASS_TO_PASS construction

Unpacks the classifier into inspectable steps. The pipeline is:

`raw runs` -> `collapse repeats` -> `quarantine flaky` -> `pivot pre/post` -> `classify`

Two assumptions are checked rather than assumed, in cells 3 and 4:
1. `agent_name` / `timestamp` are **repeat runs** of the same cell, not
   different candidate patches.
2. `patch_type` takes exactly the two values that bracket the fix.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)

# --- Schema / config: edit to match your data ---

INSTANCE: str = 'instance_id'
PATCH: str = 'patch_type'
AGENT: str = 'agent_name'
TIME: str = 'timestamp'
TEST: str = 'test_name'
PASSED: str = 'passed'

# The two patch_type values that bracket the fix.
PRE: str = 'before_patch'
POST: str = 'gold'

# Identifies a single test within a single instance.
KEY: list[str] = [INSTANCE, TEST]

CATEGORIES: list[str] = [
    'FAIL_TO_PASS',
    'PASS_TO_PASS',
    'REGRESSED',
    'BROKEN',
]

## 1. Load

In [ ]:
df = pd.read_parquet('../s3-sync/test-results/')

df[PASSED] = df[PASSED].astype(bool)

print(df.shape)
df.head()

## 2. Schema sanity check

In [ ]:
missing = [c for c in [INSTANCE, PATCH, TEST, PASSED] if c not in df.columns]
assert not missing, f'missing columns: {missing}'

labels = set(df[PATCH].unique())
assert {PRE, POST} <= labels, f'expected {PRE!r}/{POST!r}, got {labels}'

print('rows:      ', len(df))
print('instances: ', df[INSTANCE].nunique())
print('tests:     ', df[TEST].nunique())
print('agents:    ', df[AGENT].nunique())
print()
print(df[PATCH].value_counts())

## 3. Are the extra rows repeat runs, or different patches?

This is the assumption everything else rests on. If more than one agent
lands in the same `(instance, patch_type, test)` cell, those rows are
either reruns (fine, collapse them) or **different candidate patches**
(not fine — the flake filter will read genuine cross-agent disagreement
as flakiness).

If it is the latter, stop and filter to the gold/reference agent.

In [ ]:
runs = (
    df.groupby([INSTANCE, PATCH, TEST], observed=True)
    .agg(
        n_rows=(PASSED, 'size'),
        n_agents=(AGENT, 'nunique'),
    )
    .reset_index()
)

print('rows per (instance, patch_type, test) cell:')
print(runs['n_rows'].value_counts().sort_index())
print()
print('cells covered by >1 agent:', int((runs['n_agents'] > 1).sum()))

In [ ]:
# Which agents produced which side of the patch? A gold/reference run
# typically shows up as one agent covering the pre-patch side.
pd.crosstab(df[AGENT], df[PATCH])

## 4. Collapse repeat runs

A test counts as passing only if it passed in **every** run. Keeping the
lenient `any` alongside the strict `all` is what makes flakiness visible
in the next step — where they disagree, the test is unstable.

In [ ]:
status = (
    df.groupby([INSTANCE, PATCH, TEST], observed=True)[PASSED]
    .agg(all_passed='all', any_passed='any')
    .reset_index()
)

print(f'{len(df)} rows -> {len(status)} cells')
status.head()

## 5. Quarantine flaky tests

In [ ]:
status['flaky'] = status['all_passed'] != status['any_passed']

print('flaky cells:', int(status['flaky'].sum()))
print('flake rate: ', f'{status["flaky"].mean():.2%}')
status[status['flaky']].head(10)

In [ ]:
# A test that is flaky under either patch_type cannot be trusted under
# the other, so quarantine the whole (instance, test) pair.
flaky_pairs = status.loc[status['flaky'], KEY].drop_duplicates()

stable = (
    status.merge(flaky_pairs, on=KEY, how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns=['_merge', 'any_passed', 'flaky'])
)

print('quarantined pairs:', len(flaky_pairs))
print('cells:', len(status), '->', len(stable))

In [ ]:
flake_by_instance = flaky_pairs.groupby(INSTANCE)[TEST].size().sort_values()

fig, ax = plt.subplots(
    figsize=(8, max(2.0, 0.25 * len(flake_by_instance))),
)
flake_by_instance.plot.barh(ax=ax)
ax.set_xlabel('flaky tests quarantined')
ax.set_ylabel('')
ax.set_title('Flakiness per instance')
plt.tight_layout()

## 6. Pivot the pre/post verdicts side by side

`reindex` guarantees both columns exist even if the flake filter wiped
out one side entirely.

In [ ]:
wide = (
    stable.pivot(index=KEY, columns=PATCH, values='all_passed')
    .reindex(columns=[PRE, POST])
    .reset_index()
)

print('missing pre-patch verdict: ', int(wide[PRE].isna().sum()))
print('missing post-patch verdict:', int(wide[POST].isna().sum()))
wide.head()

### Resolve the missing verdicts

Two conventions get decided here, and they are the ones most worth
arguing with:

| Case | Convention |
|---|---|
| No pre-patch run | The patch **added** the test. Absent == did not pass, so it becomes a FAIL_TO_PASS. |
| No post-patch run | The patch **deleted / renamed / skipped** it. Nothing to classify — drop. |

In [ ]:
n_added = int(wide[PRE].isna().sum())
n_deleted = int(wide[POST].isna().sum())

wide[PRE] = wide[PRE].fillna(False).astype(bool)
wide = wide.dropna(subset=[POST]).copy()
wide[POST] = wide[POST].astype(bool)

print(f'added by patch (no pre run):     {n_added}')
print(f'dropped by patch (no post run):  {n_deleted}')
print(f'classifiable (instance, test)s:  {len(wide)}')
wide.head()

## 7. Classify

In [ ]:
conditions = [
    (~wide[PRE]) & wide[POST],  # FAIL_TO_PASS
    wide[PRE] & wide[POST],  # PASS_TO_PASS
    wide[PRE] & (~wide[POST]),  # REGRESSED
    (~wide[PRE]) & (~wide[POST]),  # BROKEN
]
wide['category'] = np.select(conditions, CATEGORIES, default='UNKNOWN')

assert not (wide['category'] == 'UNKNOWN').any()
print(wide['category'].value_counts())
wide.head()

In [ ]:
counts = wide['category'].value_counts().reindex(CATEGORIES).fillna(0)

fig, ax = plt.subplots(figsize=(6, 4))
counts.plot.bar(ax=ax)
ax.set_ylabel('tests')
ax.set_title('Outcome transitions, all instances')
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(counts):
    ax.text(i, v, str(int(v)), ha='center', va='bottom')
plt.tight_layout()

In [ ]:
per_instance = pd.crosstab(wide[INSTANCE], wide['category']).reindex(
    columns=CATEGORIES, fill_value=0
)

fig, ax = plt.subplots(figsize=(9, max(3.0, 0.3 * len(per_instance))))
per_instance.plot.barh(stacked=True, ax=ax)
ax.set_xlabel('tests')
ax.set_ylabel('')
ax.set_title('Outcome transitions per instance')
ax.legend(loc='lower right')
plt.tight_layout()

per_instance.head()

## 8. Build the mappings

In [ ]:
def to_mapping(frame: pd.DataFrame) -> dict[str, list[str]]:
    """Collect test names into a sorted list per instance.

    Args:
        frame: Any frame carrying the instance and test-name columns.

    Returns:
        A mapping of instance id to sorted test names. Instances with no
        matching tests are omitted.
    """
    if frame.empty:
        return {}
    grouped = frame.groupby(INSTANCE, observed=True)[TEST]
    return {str(k): sorted(v) for k, v in grouped}


f2p = to_mapping(wide[wide['category'] == 'FAIL_TO_PASS'])
p2p = to_mapping(wide[wide['category'] == 'PASS_TO_PASS'])

print('instances with >=1 F2P:', len(f2p))
print('instances with >=1 P2P:', len(p2p))
next(iter(f2p.items()))

## 9. Audit before trusting any of it

- **No F2P** means the instance is unusable: nothing in it demonstrates
  the bug was fixed.
- **A regression** means the patch broke a previously-passing test.
  Suspect the gold patch or a dirty environment before believing it.

In [ ]:
regressed = to_mapping(wide[wide['category'] == 'REGRESSED'])
no_f2p = sorted(set(df[INSTANCE].unique()) - set(f2p))

print(f'instances with NO F2P (unusable): {len(no_f2p)}')
print(no_f2p[:10])
print()
print(f'instances with regressions: {len(regressed)}')
for inst, tests in list(regressed.items())[:5]:
    print(f'  {inst}: {tests[:3]}')

In [ ]:
sizes = (
    pd.DataFrame(
        {
            'f2p': pd.Series({k: len(v) for k, v in f2p.items()}, dtype=float),
            'p2p': pd.Series({k: len(v) for k, v in p2p.items()}, dtype=float),
        }
    )
    .fillna(0)
    .astype(int)
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sizes['f2p'].plot.hist(bins=30, ax=axes[0])
axes[0].set_title('F2P tests per instance')
axes[0].set_xlabel('tests')
sizes['p2p'].plot.hist(bins=30, ax=axes[1])
axes[1].set_title('P2P tests per instance')
axes[1].set_xlabel('tests')
plt.tight_layout()

sizes.describe()

## 10. Export

In [ ]:
payload = {
    inst: {
        'FAIL_TO_PASS': f2p.get(inst, []),
        'PASS_TO_PASS': p2p.get(inst, []),
    }
    for inst in sorted(set(f2p) | set(p2p))
}

out = Path('test_splits.json')
out.write_text(json.dumps(payload, indent=2))
print(f'wrote {len(payload)} instances to {out}')